In [ ]:
# ! pip install -q langchain_community faiss-cpu langchain langchain_huggingface langgraph langchain_anthropic

In [ ]:
# Langchain Core
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_core.runnables.history import RunnableWithMessageHistory

# LangGraph
from langgraph.prebuilt import create_react_agent as langgraph_react_agent

# Langchain Community
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_community.tools import TavilySearchResults

# Langchain and External Tools
from langchain import hub
from langchain.tools.retriever import create_retriever_tool
from langchain.agents import AgentExecutor, create_react_agent
from langchain.prompts import PromptTemplate
from langchain.memory import ChatMessageHistory

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_anthropic import ChatAnthropic
from langchain_text_splitters import RecursiveCharacterTextSplitter

import os

In [ ]:
os.environ["LANGCHAIN_API_KEY"] = "YOUR_LANGSMITH_API_KEY"
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "YOUR_HUGGINGFACE_TOKEN"
os.environ["TAVILY_API_KEY"] = "YOUR_TAVILY_API_KEY"
os.environ["ANTHROPIC_API_KEY"] = "YOUR_ANTHROPIC_API_KEY"

In [ ]:
@tool
def get_retriever(URL: str, k: int = 3,
                  model_name:str="sentence-transformers/all-mpnet-base-v2"
                  ):
    """
    Carrega documentos de uma URL, divide em chunks e cria um retriever.
    """
    loader = WebBaseLoader(URL)
    docs = loader.load()

    for i, doc in enumerate(docs):
        # Limpeza mais robusta do conteúdo
        content = " ".join(doc.page_content.split())
        doc.page_content = content

    # Divisão de documentos em chunks
    splitter = RecursiveCharacterTextSplitter(separators="\n\n",
                                              chunk_size=1000,
                                              chunk_overlap=200,
                                              length_function=len)
    documents = splitter.split_documents(docs)

    # Geração de embeddings
    embeddings = HuggingFaceEmbeddings(model_name=model_name)
    vector = FAISS.from_documents(documents, embeddings)
    vector.save_local("faiss_index")

    # Criação do retriever
    retriever = vector.as_retriever(search_type="mmr",
                                    search_kwargs={"k": k})

    return retriever

/usr/local/lib/python3.10/dist-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in GetRetriever has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in get_retriever has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


In [ ]:
URL = "https://www.eb.mil.br/web/noticias/w/exercito-brasileiro-realiza-a-operacao-guararapes-2024-o-maior-exercicio-militar-do-nordeste"
retriever = get_retriever(URL)

<ipython-input-5-7ff746f711e0>:2: LangChainDeprecationWarning: The method `BaseTool.__call__` was deprecated in langchain-core 0.1.47 and will be removed in 1.0. Use :meth:`~invoke` instead.
  retriever = get_retriever(URL)
/usr/local/lib/python3.10/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models o

In [ ]:
description = """
Use esta ferramenta para acessar informações relevantes sobre
o contexto específico fornecido e responder à pergunta claramente formulada
com base nas informações obtidas.
"""
retriever_tool = create_retriever_tool(retriever,
                                       name="retriever",
                                       description=description)

search = TavilySearchResults(
    max_results=5,
    search_depth="advanced",
    include_answer=True,
    name="live_search",
    description="Use to search the latest news from the internet.",
)

tools = [
    retriever_tool,
    search,
]

In [ ]:
llm = ChatAnthropic(
    model="claude-3-haiku-20240307",
    temperature=0,
    max_tokens=1024,
    timeout=None,
    max_retries=2,
)

prompt = hub.pull("hwchase17/react")

In [ ]:
agent = create_react_agent(
    llm=llm, tools=tools, prompt=prompt, stop_sequence=True
)

agent_executor = AgentExecutor.from_agent_and_tools(
    agent=agent, tools=tools, verbose=True, handle_parsing_errors=True
)

In [ ]:
memory = ChatMessageHistory()
agent_with_chat_history = RunnableWithMessageHistory(
    agent_executor,
    # Isso é necessário porque na maioria dos cenários do mundo real, um ID de sessão é necessário
    # Não é realmente usado aqui porque estamos usando um ChatMessageHistory simples na memória
    lambda session_id: memory,
    input_messages_key="input",
    history_messages_key="chat_history",
)

In [ ]:
def get_response(query: str):
    resp = agent_with_chat_history.invoke(
        {"input": query},
        config={"configurable": {"session_id": "<foo>"}}
    )

    return resp["output"]

In [ ]:
query = "Em qual cidade foi realizada a operação?"

resp = get_response(query)